# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [36]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')
path1 = "../../datasets/day-of-week-not-scaled.csv"
paht2 = "../../datasets/dayofweek.csv"

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [37]:
df = pd.read_csv(path1)
df2 = pd.read_csv(paht2)
df["dayofweek"] = df2["dayofweek"]
X = df.drop(columns="dayofweek")
y = df["dayofweek"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [38]:
def func(model, X=X_test, y=y_test):
    y_pred = model.predict(X)
    acc = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, average="weighted")
    recall = recall_score(y, y_pred, average="weighted")
    print(f"accuracy is {acc:.5f}")
    print(f"precision is {precision:.5f}")
    print(f"recall is {recall:.5f}")

In [39]:
svc = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', probability=True, random_state=21)
svc.fit(X_train, y_train)
func(svc)

accuracy is 0.88757
precision is 0.89267
recall is 0.88757


In [40]:
tree = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)
tree.fit(X_train, y_train)
func(tree)

accuracy is 0.88757
precision is 0.89072
recall is 0.88757


In [41]:
forest = RandomForestClassifier(class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100, random_state=21)
forest.fit(X_train, y_train)
func(forest)

accuracy is 0.92899
precision is 0.93035
recall is 0.92899


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [42]:
voting = VotingClassifier(estimators=[("svc", svc), ("tree", tree), ("forest", forest)], voting="soft")
voting.fit(X_train, y_train)
func(voting)

accuracy is 0.91716
precision is 0.91945
recall is 0.91716


In [43]:
weights_lst = [[1, 2, 3], [2, 1, 1], [1, 2, 1], [1, 2, 2], [4, 1, 4]]
for weights in weights_lst:
    cur_voting = VotingClassifier(estimators=[("svc", svc), ("tree", tree), ("forest", forest)], voting="soft", weights=weights)
    cur_voting.fit(X_train, y_train)
    print(f"weights={weights}")
    func(cur_voting)
    print()

weights=[1, 2, 3]
accuracy is 0.91124
precision is 0.91344
recall is 0.91124

weights=[2, 1, 1]
accuracy is 0.91124
precision is 0.91366
recall is 0.91124

weights=[1, 2, 1]
accuracy is 0.88757
precision is 0.89072
recall is 0.88757

weights=[1, 2, 2]
accuracy is 0.89941
precision is 0.90159
recall is 0.89941

weights=[4, 1, 4]
accuracy is 0.92012
precision is 0.92171
recall is 0.92012



## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [44]:
n = [10, 20, 30, 40, 50, 75, 100]
for j in n:
    cur_bagging = BaggingClassifier(estimator=svc, n_estimators=j, random_state=21)
    cur_bagging.fit(X_train, y_train)
    print(f"n_estimators={j}")
    func(cur_bagging)
    print()

n_estimators=10
accuracy is 0.88757
precision is 0.89182
recall is 0.88757

n_estimators=20
accuracy is 0.89941
precision is 0.90369
recall is 0.89941

n_estimators=30
accuracy is 0.89941
precision is 0.90351
recall is 0.89941

n_estimators=40
accuracy is 0.90533
precision is 0.90850
recall is 0.90533

n_estimators=50
accuracy is 0.90828
precision is 0.91091
recall is 0.90828

n_estimators=75
accuracy is 0.90828
precision is 0.91091
recall is 0.90828

n_estimators=100
accuracy is 0.90828
precision is 0.91091
recall is 0.90828



## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [46]:
splits = [2, 3, 4, 5, 6, 7]
svc = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', probability=True, random_state=21)
tree = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)
forest = RandomForestClassifier(class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100, random_state=21)

for n in splits:
    cv = StratifiedKFold(n_splits=n, shuffle=True, random_state=21)
    cur_stacking = StackingClassifier(estimators=[("svc", svc), ("tree", tree), ("forest", forest)],
                                  final_estimator=LogisticRegression(solver='lbfgs'), 
                                  cv=cv, passthrough=True)
    cur_stacking.fit(X_train, y_train)
    print(f"n_splits={n}")
    func(cur_stacking, X_test, y_test)
    print()

n_splits=2
accuracy is 0.92308
precision is 0.92583
recall is 0.92308

n_splits=3
accuracy is 0.94083
precision is 0.94177
recall is 0.94083

n_splits=4
accuracy is 0.92899
precision is 0.93016
recall is 0.92899

n_splits=5
accuracy is 0.93787
precision is 0.93974
recall is 0.93787

n_splits=6
accuracy is 0.92604
precision is 0.92773
recall is 0.92604

n_splits=7
accuracy is 0.93491
precision is 0.93616
recall is 0.93491



## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [ ]:
voting = VotingClassifier(estimators=[("svc", svc), ("tree", tree), ("forest", forest)], voting="soft", weights=[4, 1, 4])
voting.fit(X_train, y_train)
func(voting, X=X_test, y=y_test)

accuracy is 0.92012

recall is 0.92012
precision is 0.92171
recall is 0.92012


In [ ]:
best_model = voting
y_pred = best_model.predict(X_test)

In [ ]:
res = pd.DataFrame({"true": y_test, "pred": y_pred})
user_cols = [col for col in X_test.columns if col.startswith("uid_")]
res["user"] = X_test[user_cols].idxmax(axis=1)
labname_cols = [col for col in X_test.columns if col.startswith("labname_")]
res["labname"] = X_test[labname_cols].idxmax(axis=1)

In [ ]:
for weekday in res["true"].unique():
    class_data = res[res["true"] == weekday]
    total = len(class_data)
    errors = (class_data["true"] != class_data["pred"]).sum()
    e_rate = errors / total * 100
    print(f"{weekday}: {e_rate:.2f}%")

1: 12.73%
5: 7.41%
6: 7.04%
3: 3.75%
2: 6.67%
4: 4.76%
0: 18.52%


In [ ]:
for user in res["user"].unique():
    data = res[res["user"] == user]
    total = len(data)
    errors = (data["true"] != data["pred"]).sum()
    e_rate = errors / total * 100
    print(f"{user}: {e_rate:.2f}%")

uid_user_14: 6.45%
uid_user_2: 7.14%
uid_user_12: 0.00%
uid_user_20: 0.00%
uid_user_28: 12.50%
uid_user_25: 0.00%
uid_user_17: 28.57%
uid_user_31: 0.00%
uid_user_27: 16.67%
uid_user_15: 0.00%
uid_user_4: 11.11%
uid_user_30: 12.50%
uid_user_24: 9.09%
uid_user_26: 0.00%
uid_user_3: 14.29%
uid_user_10: 0.00%
uid_user_29: 9.09%
uid_user_13: 11.76%
uid_user_22: 100.00%
uid_user_21: 0.00%
uid_user_8: 0.00%
uid_user_19: 21.05%
uid_user_1: 0.00%
uid_user_6: 50.00%
uid_user_23: 0.00%
uid_user_16: 20.00%
uid_user_18: 16.67%


In [ ]:
for laba in res["labname"].unique():
    data = res[res["labname"] == laba]
    total = len(data)
    errors = (data["true"] != data["pred"]).sum()
    e_rate = errors / total * 100
    print(f"{laba}: {e_rate:.2f}%")

labname_project1: 3.76%
labname_laba04s: 16.00%
labname_laba05: 0.00%
labname_laba04: 25.71%
labname_code_rvw: 7.69%
labname_laba06: 22.22%
labname_laba06s: 13.33%
labname_lab05s: 16.67%
labname_lab03s: 0.00%
labname_lab03: 100.00%


In [ ]:
joblib.dump(best_model, "model.pkl")

['model.pkl']